In [4]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [5]:
import json

from src.config import RAW_DOCUMENTS_DIR, TOP_K
from src.loader import load_documents
from src.pipeline import build_knowledge_base
from src.retriever import retrieve

/home/prashant/Projects/rag-knowledge-assistant/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
import json
from pathlib import Path

project_root = Path.cwd().parent

evaluation_file = (
    project_root / "evaluation" / "questions.json"
)

with open(
    evaluation_file,
    "r",
    encoding="utf-8",
) as file:
    questions = json.load(file)

len(questions)

4

In [9]:
documents = load_documents(
    RAW_DOCUMENTS_DIR
)

model, index, chunks = build_knowledge_base(
    documents
)

print(f"Documents: {len(documents)}")
print(f"Chunks: {len(chunks)}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6323.38it/s]


Documents: 1
Chunks: 1


In [10]:
question = questions[0]["question"]

results = retrieve(
    query=question,
    model=model,
    index=index,
    chunks=chunks,
    top_k=3,
)

for result in results:
    print(f"Score: {result['score']:.3f}")
    print(result["text"])
    print("-" * 60)

Score: 0.728
Company Annual Leave Policy Employees receive 30 days of annual leave per calendar year. Employees should request annual leave through the company's HR system. Unused annual leave may be carried over to the following year, subject to company policy. Employees should normally request leave at least two weeks in advance. Remote Work Policy Employees may work remotely up to three days per week with manager approval. Employees working remotely must remain available during their agreed working hours.
------------------------------------------------------------


In [11]:
expected_text = questions[0]["expected_text"]

retrieved_text = " ".join(
    result["text"]
    for result in results
)

expected_text.lower() in retrieved_text.lower()

True

In [12]:
results_summary = []

for item in questions:

    results = retrieve(
        query=item["question"],
        model=model,
        index=index,
        chunks=chunks,
        top_k=TOP_K,
    )

    retrieved_text = " ".join(
        result["text"]
        for result in results
    )

    success = (
        item["expected_text"].lower()
        in retrieved_text.lower()
    )

    results_summary.append(
        {
            "question": item["question"],
            "success": success,
        }
    )

results_summary

[{'question': 'How many days of annual leave do employees receive?',
  'success': True},
 {'question': 'How far in advance should employees normally request leave?',
  'success': True},
 {'question': 'How many days per week can employees work remotely?',
  'success': True},
 {'question': 'What must remote employees do during their working hours?',
  'success': True}]

In [13]:
successful = sum(
    result["success"]
    for result in results_summary
)

total = len(results_summary)

recall_at_3 = successful / total

print(f"Recall@3: {recall_at_3:.2%}")

Recall@3: 100.00%


In [14]:
k_values = [1, 3, 5]

for k in k_values:

    successful = 0

    for item in questions:

        results = retrieve(
            query=item["question"],
            model=model,
            index=index,
            chunks=chunks,
            top_k=k,
        )

        retrieved_text = " ".join(
            result["text"]
            for result in results
        )

        if item["expected_text"].lower() in retrieved_text.lower():
            successful += 1

    recall = successful / len(questions)

    print(f"Recall@{k}: {recall:.2%}")

Recall@1: 100.00%
Recall@3: 100.00%
Recall@5: 100.00%
